 ══════════════════════════════════════════════════════════════════════════════
## APRENDIZAJE AUTOMÁTICO II — Tecnicatura Universitaria en IA (UNR - FCEIA)
## Trabajo Práctico N°2 — Ejercicio 3: Reconocimiento de Comandos de Voz con CRNN
══════════════════════════════════════════════════════════════════════════════

**Integrantes:** Sebastian Palacio, Juana Chies Doumecq

**Objetivo:** Construir, entrenar y optimizar una Red Neuronal Convolucional-Recurrente (CRNN) para reconocer comandos de voz a partir de grabaciones de audio de un segundo del dataset Google Speech Commands V2.

**Dataset:** [Google Speech Commands V2](https://www.kaggle.com/datasets/sylkaladin/speech-commands-v2)

**Pipeline:**
1. Cargar audios `.wav` → Mel Spectrogram (librosa)
2. Padding/truncado + normalización
3. Ventana deslizante → secuencia de ventanas (F, W)
4. CNN encoder → vector de características por ventana
5. GRU/LSTM bidireccional → clasificación many-to-one


## 1. Configuración y reproducibilidad

In [ ]:
# ─── LIBRERÍAS ESTÁNDAR ───────────────────────────────────────────────────────
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ─── VISUALIZACIÓN ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ─── AUDIO ────────────────────────────────────────────────────────────────────
import librosa
import librosa.display

# ─── PYTORCH ──────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

print("✅ Librerías importadas correctamente")
print(f"   PyTorch versión: {torch.__version__}")
print(f"   librosa versión: {librosa.__version__}")


### 1.1 Semilla fija para reproducibilidad

Los algoritmos de Deep Learning usan números aleatorios en inicialización de pesos, 
Dropout y orden de batches. Fijar todas las semillas garantiza resultados reproducibles.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
print(f"✅ Semilla fijada: {SEED}")


### 1.2 Configuración del dispositivo

PyTorch detecta automáticamente el hardware disponible (CUDA, MPS, CPU) 
y usa el más rápido para ejecutar el entrenamiento.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("=" * 50)
print("CONFIGURACIÓN DEL ENTORNO")
print("=" * 50)
print(f"  Dispositivo : {device}")
if device.type == "cuda":
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 50)


## 2. Carga e inspección del dataset

### 2.1 Configuración y selección de comandos

El dataset Google Speech Commands V2 contiene grabaciones de ~1 segundo de muchos 
hablantes distintos en condiciones acústicas reales. Usamos las 10 clases sugeridas 
por el enunciado: `yes, no, up, down, left, right, on, off, stop, go`.

Estos comandos tienen variedad fonética interesante:
- Monosílabos cortos: *yes, no, up, on, go, off*
- Bisílabos: *down, left, right, stop*
- Pares confundibles fonéticamente: *yes/no*, *up/down*, *left/right*, *on/off*, *stop/go*


In [ ]:
DATASET_ROOT = Path("./Datasets/problema 3")

print(f"Dataset root : {DATASET_ROOT}")
print(f"Existe       : {DATASET_ROOT.exists()}")

COMANDOS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
N_CLASES = len(COMANDOS)
# Mapeo clase → índice numérico para CrossEntropyLoss
CLASE_A_IDX = {c: i for i, c in enumerate(COMANDOS)}
IDX_A_CLASE = {i: c for c, i in CLASE_A_IDX.items()}

print(f"\nComandos seleccionados ({N_CLASES}):")
for i, cmd in enumerate(COMANDOS):
    print(f"  {i:2d}: {cmd}")


### 2.2 Leer splits oficiales

El dataset provee `validation_list.txt` y `testing_list.txt` con particiones que 
garantizan separación estricta por hablante: ningún hablante del test aparece en train.


In [ ]:
# Verificar que todas las carpetas existen
carpetas = sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])
faltantes = [c for c in COMANDOS if c not in carpetas]
if faltantes:
    print("⚠️  ADVERTENCIA — Clases no encontradas:", faltantes)
else:
    print("✅ Todas las clases encontradas")

# Leer listas oficiales de splits
VAL_LIST  = DATASET_ROOT / "validation_list.txt"
TEST_LIST = DATASET_ROOT / "testing_list.txt"

with open(VAL_LIST,  "r") as f:
    validation_files = set(line.strip() for line in f)

with open(TEST_LIST, "r") as f:
    testing_files = set(line.strip() for line in f)

print(f"\nAudios en validation_list : {len(validation_files):,}")
print(f"Audios en testing_list    : {len(testing_files):,}")


### 2.3 Construcción de Train / Val / Test

Asignamos cada archivo `.wav` a su split correspondiente según las listas oficiales.


In [ ]:
train_samples, val_samples, test_samples = [], [], []

for comando in COMANDOS:
    carpeta = DATASET_ROOT / comando
    for archivo in carpeta.glob("*.wav"):
        relative_path = f"{comando}/{archivo.name}"
        registro = {"path": archivo, "label": comando, "label_idx": CLASE_A_IDX[comando]}
        if   relative_path in validation_files: val_samples.append(registro)
        elif relative_path in testing_files:    test_samples.append(registro)
        else:                                   train_samples.append(registro)

# Resumen
cnt_train = Counter(x["label"] for x in train_samples)
cnt_val   = Counter(x["label"] for x in val_samples)
cnt_test  = Counter(x["label"] for x in test_samples)

print(f"\n{'Clase':<12} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 40)
for clase in COMANDOS:
    print(f"{clase:<12}{cnt_train[clase]:>8}{cnt_val[clase]:>8}{cnt_test[clase]:>8}")
print("-" * 40)
print(f"{'TOTAL':<12}{len(train_samples):>8}{len(val_samples):>8}{len(test_samples):>8}")


## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Balance de clases

Antes de entrenar verificamos que las clases estén balanceadas para no necesitar 
ponderación especial en la función de pérdida.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
splits = [("Train", cnt_train), ("Validación", cnt_val), ("Test", cnt_test)]

for ax, (nombre, cnt) in zip(axes, splits):
    valores = [cnt[c] for c in COMANDOS]
    barras  = ax.bar(COMANDOS, valores, color=sns.color_palette("husl", N_CLASES))
    ax.set_title(f"{nombre} ({sum(valores):,} muestras)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Comando")
    ax.set_ylabel("Cantidad de audios")
    ax.tick_params(axis='x', rotation=45)
    # Anotar valores
    for barra, val in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 10,
                str(val), ha='center', va='bottom', fontsize=8)

plt.suptitle("Distribución de clases por split", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("📊 El dataset está balanceado: todas las clases tienen cantidades similares.")


## 4. Preprocesamiento: Del audio al Mel Spectrogram

### 4.1 Conversión Audio → Mel Spectrogram

Los Mel Spectrogramas representan la energía de la señal en función del tiempo (eje X) 
y la frecuencia en escala Mel (eje Y). La escala Mel imita la percepción auditiva humana 
comprimiendo frecuencias altas. El color indica intensidad en dB.

**Parámetros fijados según el enunciado:**
- `sr=16000`: frecuencia de muestreo del dataset
- `n_mels=64`: resolución frecuencial
- `n_fft=1024`: tamaño de la FFT (resolución espectral)
- `hop_length=512`: paso entre frames (~32 ms entre columnas)

Con `sr=16000` y `hop_length=512`, cada segundo de audio produce `16000/512 ≈ 32` frames.
Usamos un `TARGET_LEN=32` para normalizar todos los espectrogramas.


In [ ]:
# ── Hiperparámetros de preprocesamiento ──────────────────────────────────────
SR          = 16000   # frecuencia de muestreo (fija por el dataset)
N_MELS      = 64      # bandas de frecuencia Mel
N_FFT       = 1024    # tamaño de la FFT
HOP_LENGTH  = 512     # paso entre frames
TARGET_LEN  = 32      # número de frames temporales (≈ 1 segundo a sr=16000, hop=512)

def audio_a_mel(path, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                hop_length=HOP_LENGTH, target_len=TARGET_LEN):
    """
    Carga un archivo .wav y retorna su Mel Spectrogram normalizado en escala dB.

    Pasos:
      1. Cargar y resamplear a sr Hz.
      2. Calcular Mel Spectrogram → shape (n_mels, T).
      3. Convertir a dB (escala logarítmica).
      4. Padding con ceros si T < target_len; truncar si T > target_len.
         Garantiza que todos los espectrogramas tengan el mismo ancho temporal.

    Retorna:
      mel_db: ndarray float32 de shape (n_mels, target_len)
    """
    # Cargar audio y resamplear a 16 kHz
    y, _ = librosa.load(str(path), sr=sr)

    # Calcular Mel Spectrogram: shape (n_mels, T)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )

    # Convertir a decibeles: escala logarítmica que imita la percepción auditiva
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Padding o truncado para que todos tengan el mismo target_len
    T = mel_db.shape[1]
    if T < target_len:
        # Padding a la derecha con ceros (silencio = energía mínima)
        pad = target_len - T
        mel_db = np.pad(mel_db, ((0, 0), (0, pad)), mode='constant',
                        constant_values=mel_db.min())
    else:
        mel_db = mel_db[:, :target_len]

    return mel_db.astype(np.float32)  # shape: (N_MELS, TARGET_LEN)

# Prueba rápida con un audio de ejemplo
ejemplo_path = train_samples[0]["path"]
ejemplo_mel   = audio_a_mel(ejemplo_path)
print(f"✅ Mel Spectrogram generado")
print(f"   Archivo : {ejemplo_path.name}")
print(f"   Shape   : {ejemplo_mel.shape}  (n_mels={N_MELS}, target_len={TARGET_LEN})")
print(f"   Min dB  : {ejemplo_mel.min():.2f}")
print(f"   Max dB  : {ejemplo_mel.max():.2f}")


### 4.2 Normalización global

Calculamos media y desviación estándar **solo sobre el conjunto de train** 
y las aplicamos a train, val y test. Esto evita data leakage.


In [ ]:
print("Calculando estadísticas de normalización sobre train...")
print("(Esto puede tardar unos minutos la primera vez)")

# Calcular media y std solo sobre train para evitar data leakage
all_mels  = []
n_muestras_norm = min(3000, len(train_samples))  # subconjunto para eficiencia

for sample in random.sample(train_samples, n_muestras_norm):
    mel = audio_a_mel(sample["path"])
    all_mels.append(mel)

all_mels_array = np.stack(all_mels)  # (N, n_mels, target_len)
MEL_MEAN = float(all_mels_array.mean())
MEL_STD  = float(all_mels_array.std()) + 1e-8  # +epsilon para evitar división por cero

print(f"\n✅ Estadísticas calculadas sobre {n_muestras_norm} audios de train:")
print(f"   Media : {MEL_MEAN:.4f} dB")
print(f"   Std   : {MEL_STD:.4f} dB")


### 4.3 Ventana deslizante

Antes de pasar el espectrograma a la CNN, lo segmentamos en ventanas temporales 
con solapamiento. Cada ventana (F, W) representa una región acústica del audio.

Con `window_size=16` y `hop=8`, el solapamiento es del 50%, garantizando que cada 
transición fonética aparezca en al menos dos ventanas consecutivas.
Un espectrograma de 32 frames produce `(32 - 16) / 8 + 1 = 3` ventanas.


In [ ]:
# ── Hiperparámetros de ventana (configuración base E1) ───────────────────────
WINDOW_SIZE = 16   # ancho de cada ventana en frames temporales
HOP_WIN     = 8    # desplazamiento entre ventanas (solapamiento = window_size - hop)

def extraer_ventanas(mel, window_size=WINDOW_SIZE, hop=HOP_WIN):
    """
    Divide un espectrograma (F, T) en ventanas con solapamiento.

    Args:
        mel         : array (F, T)
        window_size : ancho de cada ventana en frames temporales
        hop         : desplazamiento entre ventanas (hop < window_size → solapamiento)

    Returns:
        ventanas : array (N, F, window_size) con N ventanas
    """
    F, T = mel.shape
    ventanas = []
    for start in range(0, T - window_size + 1, hop):
        ventana = mel[:, start: start + window_size]  # (F, W)
        ventanas.append(ventana)
    return np.stack(ventanas)  # (N, F, W)

# Prueba con el espectrograma de ejemplo
ventanas_ejemplo = extraer_ventanas(ejemplo_mel)
print(f"✅ Ventana deslizante:")
print(f"   Espectrograma entrada : {ejemplo_mel.shape}")
print(f"   window_size={WINDOW_SIZE}, hop={HOP_WIN}")
print(f"   Overlap               : {WINDOW_SIZE - HOP_WIN} frames ({(WINDOW_SIZE-HOP_WIN)/WINDOW_SIZE*100:.0f}%)")
print(f"   Ventanas generadas    : {ventanas_ejemplo.shape}  (N_ventanas, F, W)")
print(f"   Shape CNN (1 muestra) : ({ventanas_ejemplo.shape[0]}, 1, {N_MELS}, {WINDOW_SIZE})")


## 5. Análisis de Espectrogramas (EDA visual obligatorio)

### 5.1 Visualización de espectrogramas por clase

Antes de entrenar, analizamos al menos 8 espectrogramas para entender la estructura
temporal y frecuencial de cada comando.

**Qué observar:**
- **Eje X (tiempo):** duración de la señal útil, presencia de silencios
- **Eje Y (frecuencia Mel):** rango de frecuencias con mayor energía
- **Color (energía en dB):** zonas brillantes = alta energía = sonido activo


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

# 4 instancias de la misma clase ("yes") + 4 clases distintas
clases_mostrar = ["yes", "yes", "yes", "yes", "no", "up", "down", "stop"]

for ax, clase in zip(axes, clases_mostrar):
    # Elegir un audio aleatorio de la clase
    muestra = random.choice([s for s in train_samples if s["label"] == clase])
    mel = audio_a_mel(muestra["path"])

    img = librosa.display.specshow(
        mel, sr=SR, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel",
        ax=ax, cmap="viridis"
    )
    ax.set_title(f'"{clase}" — {muestra["path"].name}', fontsize=9)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Mel)")

plt.suptitle(
    "Panel de Mel Spectrogramas\n"
    "Fila 1: 4 instancias de \'yes\' (variabilidad intra-clase) | "
    "Fila 2: clases distintas (variabilidad inter-clase)",
    fontsize=12, fontweight='bold'
)
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6)
plt.tight_layout()
plt.show()


### 5.2 Análisis cualitativo de los espectrogramas

**Eje horizontal (tiempo):**  
La señal útil ocupa típicamente entre 0.3 y 0.8 segundos del audio de 1 segundo total.
Es frecuente observar silencio en los primeros y últimos ~100 ms: los hablantes 
no comienzan ni terminan el comando exactamente en el borde del archivo.

**Eje vertical (frecuencia Mel):**  
La energía de los comandos monosílabos (*yes, no, up, on, go*) se concentra 
principalmente entre 200 Hz y 4000 Hz. Los comandos con fricativas (*stop*, /s/) 
muestran energía adicional en frecuencias altas (>4000 Hz).

**Color (energía en dB):**  
Las zonas de colores cálidos (amarillo-verde) corresponden a fonemas vocálicos 
y fricativos sonoros. Las bandas horizontales brillantes son los formantes vocálicos 
(F1 y F2), cuya posición relativa distingue vocales entre sí.

**Variabilidad intra-clase (4 instancias de "yes"):**  
Aunque todos dicen la misma palabra, los spectrogramas varían en duración, 
amplitud y posición temporal de los formantes. Esto se debe a diferencias de 
velocidad, tono y acento entre hablantes.

**Diferencias inter-clase:**  
- *yes* vs *no*: "yes" (/jɛs/) muestra energía en frecuencias medias-altas por la /s/ final; 
  "no" (/noʊ/) concentra energía en frecuencias bajas-medias por la vocal /oʊ/.
- *up* vs *down*: "up" es muy breve; "down" tiene mayor duración con transición /aʊ/ → /n/.
- *stop* es distinguible visualmente por la energía de alta frecuencia de la /s/ inicial.


## 6. Dataset y DataLoaders de PyTorch

In [ ]:
class SpeechCommandsDataset(Dataset):
    """
    Dataset de PyTorch para Google Speech Commands.

    Para cada audio:
      1. Carga y convierte a Mel Spectrogram (F, T).
      2. Normaliza con media y std del train set.
      3. Extrae ventanas deslizantes → (N, F, W).
      4. Agrega canal monocanal para la CNN → (N, 1, F, W).

    El tensor de salida tiene shape (N_ventanas, 1, N_MELS, WINDOW_SIZE),
    que es lo que espera la CRNN: N_ventanas pasos de secuencia,
    cada uno como una imagen monocanal de tamaño (N_MELS, WINDOW_SIZE).
    """

    def __init__(self, samples, mel_mean, mel_std,
                 window_size=WINDOW_SIZE, hop=HOP_WIN,
                 n_mels=N_MELS, target_len=TARGET_LEN,
                 augment=False):
        self.samples     = samples
        self.mel_mean    = mel_mean
        self.mel_std     = mel_std
        self.window_size = window_size
        self.hop         = hop
        self.n_mels      = n_mels
        self.target_len  = target_len
        self.augment     = augment  # Data augmentation solo en train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # 1. Cargar audio → Mel Spectrogram (n_mels, target_len)
        mel = audio_a_mel(
            sample["path"],
            n_mels=self.n_mels,
            target_len=self.target_len
        )

        # 2. Data augmentation (solo en train)
        if self.augment:
            # SpecAugment ligero: enmascarar bandas de frecuencia o frames temporales
            mel = self._spec_augment(mel)

        # 3. Normalización: (x - mean) / std → media ≈ 0, std ≈ 1
        mel = (mel - self.mel_mean) / self.mel_std

        # 4. Extraer ventanas deslizantes → (N_ventanas, n_mels, window_size)
        ventanas = extraer_ventanas(mel, self.window_size, self.hop)

        # 5. Agregar canal monocanal → (N_ventanas, 1, n_mels, window_size)
        ventanas = ventanas[:, np.newaxis, :, :]

        # 6. Convertir a tensor float32
        x = torch.from_numpy(ventanas).float()
        y = torch.tensor(sample["label_idx"], dtype=torch.long)

        return x, y

    def _spec_augment(self, mel):
        """
        Versión simplificada de SpecAugment (Park et al., 2019).
        Enmascara una banda de frecuencia y un segmento temporal aleatorios.
        Ayuda a regularizar el modelo para que no dependa de una sola frecuencia o instante.
        """
        mel = mel.copy()
        F, T = mel.shape

        # Máscara de frecuencia: pone a cero f bandas consecutivas
        f_mask = random.randint(0, min(10, F // 4))
        f0     = random.randint(0, F - f_mask)
        mel[f0: f0 + f_mask, :] = mel.min()

        # Máscara temporal: pone a cero t frames consecutivos
        t_mask = random.randint(0, min(4, T // 4))
        t0     = random.randint(0, T - t_mask)
        mel[:, t0: t0 + t_mask] = mel.min()

        return mel


# ── Crear los tres datasets ───────────────────────────────────────────────────
ds_train = SpeechCommandsDataset(train_samples, MEL_MEAN, MEL_STD, augment=True)
ds_val   = SpeechCommandsDataset(val_samples,   MEL_MEAN, MEL_STD, augment=False)
ds_test  = SpeechCommandsDataset(test_samples,  MEL_MEAN, MEL_STD, augment=False)

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE  = 64
N_WORKERS   = 2

loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=N_WORKERS, pin_memory=True)
loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=N_WORKERS, pin_memory=True)
loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=N_WORKERS, pin_memory=True)

# Verificar shapes
x_ejemplo, y_ejemplo = ds_train[0]
print("✅ Dataset y DataLoaders creados")
print(f"   Train : {len(ds_train):,} muestras  |  {len(loader_train):,} batches")
print(f"   Val   : {len(ds_val):,}  muestras  |  {len(loader_val):,} batches")
print(f"   Test  : {len(ds_test):,} muestras  |  {len(loader_test):,} batches")
print(f"\n   Shape tensor por muestra: {x_ejemplo.shape}  (N_ventanas, 1, F, W)")
print(f"   N_ventanas = {x_ejemplo.shape[0]}, F = {x_ejemplo.shape[2]}, W = {x_ejemplo.shape[3]}")


## 7. Arquitectura CRNN

### 7.1 Diseño de la arquitectura base (E1)

La CRNN sigue un esquema **many-to-one**:

1. **CNN Encoder:** procesa cada ventana `(1, F, W)` de forma independiente → vector de 128 features.
   Equivalente a `TimeDistributed(CNN)` en Keras.
2. **GRU bidireccional:** recibe la secuencia de vectores `(N_ventanas, 128)` y modela
   cómo los patrones acústicos evolucionan en el tiempo.
3. **Clasificador:** el último estado oculto `(hidden*2,)` se proyecta a `N_CLASES` logits.

**¿Por qué bidireccional?**  
En un comando como "stop", la /t/ final da contexto crucial para distinguirlo de "top".
Una GRU bidireccional puede usar contexto futuro (segunda pasada hacia atrás) para 
reforzar la clasificación.


In [ ]:
class CNNEncoder(nn.Module):
    """
    Codificador CNN: procesa UNA ventana (batch, 1, F, W) → vector (batch, 128).

    Arquitectura base E1:
      Bloque 1: Conv(1→32) + BN + ReLU + MaxPool(2) → (32, F/2, W/2)
      Bloque 2: Conv(32→64) + BN + ReLU + MaxPool(2) → (64, F/4, W/4)
      Flatten + Linear → 128

    La salida de 128 dimensiones es la representación comprimida de esa
    ventana temporal: resume qué patrones acústicos están presentes en ella.
    """

    def __init__(self, n_mels=N_MELS, window_size=WINDOW_SIZE,
                 dropout=0.0, n_filters=(32, 64)):
        super().__init__()

        f1, f2 = n_filters

        self.cnn = nn.Sequential(
            # Bloque 1
            nn.Conv2d(1, f1, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(f1),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (f1, F/2, W/2)

            # Bloque 2
            nn.Conv2d(f1, f2, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(f2),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (f2, F/4, W/4)
        )

        # Dropout dentro del encoder (regularización)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        # Tamaño de la salida aplanada antes de la FC
        # Las dos MaxPool(2) dividen F y W por 4 cada una
        self.out_size = f2 * (n_mels // 4) * (window_size // 4)

        # Proyección a 128 dimensiones (representación compacta)
        self.fc = nn.Linear(self.out_size, 128)

    def forward(self, x):
        # x: (batch, 1, F, W)
        x = self.cnn(x)        # (batch, 64, F/4, W/4)
        x = self.dropout(x)
        x = x.flatten(1)       # (batch, out_size)
        return self.fc(x)      # (batch, 128)


class CRNN(nn.Module):
    """
    Red Convolucional-Recurrente para clasificación de comandos de voz.

    Entrada : (batch, N_ventanas, 1, F, W)
    Salida  : (batch, N_CLASES) — logits sin Softmax (para CrossEntropyLoss)

    El forward aplica el CNNEncoder a cada ventana de forma paralela
    (reshape batch*N → encoder → reshape back), luego la RNN procesa
    la secuencia resultante.
    """

    def __init__(self, n_mels=N_MELS, window_size=WINDOW_SIZE, n_classes=N_CLASES,
                 rnn_hidden=128, rnn_layers=2, rnn_type='GRU',
                 bidirectional=True, rnn_dropout=0.3,
                 cnn_dropout=0.0, cnn_filters=(32, 64),
                 aggregation='last'):
        super().__init__()

        self.encoder     = CNNEncoder(n_mels, window_size, cnn_dropout, cnn_filters)
        self.rnn_hidden  = rnn_hidden
        self.bidirectional = bidirectional
        self.aggregation = aggregation  # 'last' o 'mean'

        # Selección de tipo de celda recurrente
        rnn_class = nn.GRU if rnn_type == 'GRU' else nn.LSTM
        self.rnn = rnn_class(
            input_size  = 128,           # tamaño de salida del CNNEncoder
            hidden_size = rnn_hidden,
            num_layers  = rnn_layers,
            batch_first = True,          # (batch, seq, features)
            bidirectional = bidirectional,
            dropout     = rnn_dropout if rnn_layers > 1 else 0.0
        )

        # Factor 2 si bidireccional (concatena estados forward y backward)
        factor = 2 if bidirectional else 1
        self.dropout_cls = nn.Dropout(p=0.3)
        self.fc = nn.Linear(rnn_hidden * factor, n_classes)

    def forward(self, x):
        # x: (batch, N, 1, F, W)
        batch, N, C, F, W = x.shape

        # Procesar todas las ventanas de todos los audios del batch en paralelo
        x = x.view(batch * N, C, F, W)  # (batch*N, 1, F, W)
        x = self.encoder(x)             # (batch*N, 128)
        x = x.view(batch, N, -1)        # (batch, N, 128) — restaurar secuencia

        # RNN sobre la secuencia de vectores
        out, _ = self.rnn(x)            # (batch, N, hidden * factor)

        # Estrategia de agregación
        if self.aggregation == 'last':
            x = out[:, -1, :]           # último estado → (batch, hidden * factor)
        else:  # 'mean'
            x = out.mean(dim=1)         # promedio sobre todos los pasos

        x = self.dropout_cls(x)
        return self.fc(x)               # (batch, n_classes)


# Verificar con un batch de ejemplo
modelo_prueba = CRNN().to(device)
x_batch_prueba = torch.randn(4, x_ejemplo.shape[0], 1, N_MELS, WINDOW_SIZE).to(device)
out_prueba = modelo_prueba(x_batch_prueba)

print("✅ Arquitectura CRNN verificada")
print(f"   Entrada : {x_batch_prueba.shape}  (batch, N_ventanas, 1, F, W)")
print(f"   Salida  : {out_prueba.shape}      (batch, N_clases)")

total_params = sum(p.numel() for p in modelo_prueba.parameters() if p.requires_grad)
print(f"   Parámetros entrenables: {total_params:,}")
del modelo_prueba


## 8. Loop de entrenamiento y utilidades

In [ ]:
def entrenar_crnn(modelo, loader_train, loader_val, loader_test,
                  epochs=30, patience=8, lr=1e-3,
                  nombre='CRNN', guardar_ckpt=True):
    """
    Loop de entrenamiento para CRNN con:
    - CrossEntropyLoss + Adam
    - ReduceLROnPlateau scheduler
    - Early stopping por val_loss
    - Guardado de checkpoint del mejor modelo
    - Evaluación final en test

    Retorna historial con métricas por época + accuracy en test.
    """
    criterio    = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=lr, weight_decay=1e-4)
    scheduler   = optim.lr_scheduler.ReduceLROnPlateau(
        optimizador, mode='min', factor=0.5, patience=4, min_lr=1e-6
    )

    historial = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    mejor_val_loss = float('inf')
    mejor_pesos    = None
    espera         = 0

    print(f"\n🚀 Entrenando {nombre} | epochs={epochs} | patience={patience} | lr={lr}")
    print(f"   {'Época':>6} | {'Train Loss':>10} | {'Val Loss':>10} | "
          f"{'Train Acc':>10} | {'Val Acc':>9} | {'LR':>8}")
    print("   " + "─" * 66)

    for epoca in range(1, epochs + 1):
        # ── FASE TRAIN ────────────────────────────────────────────────────────
        modelo.train()
        train_loss_total = 0.0
        train_correctos  = 0
        train_total      = 0

        for x_batch, y_batch in loader_train:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizador.zero_grad()
            logits = modelo(x_batch)
            loss   = criterio(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            optimizador.step()

            train_loss_total += loss.item() * x_batch.size(0)
            preds             = logits.argmax(dim=1)
            train_correctos  += (preds == y_batch).sum().item()
            train_total      += x_batch.size(0)

        train_loss = train_loss_total / train_total
        train_acc  = train_correctos  / train_total

        # ── FASE VALIDACIÓN ───────────────────────────────────────────────────
        modelo.eval()
        val_loss_total = 0.0
        val_correctos  = 0
        val_total      = 0

        with torch.no_grad():
            for x_val, y_val in loader_val:
                x_val   = x_val.to(device)
                y_val   = y_val.to(device)
                logits_v = modelo(x_val)
                loss_v   = criterio(logits_v, y_val)
                val_loss_total += loss_v.item() * x_val.size(0)
                val_correctos  += (logits_v.argmax(1) == y_val).sum().item()
                val_total      += x_val.size(0)

        val_loss = val_loss_total / val_total
        val_acc  = val_correctos  / val_total

        historial['train_loss'].append(train_loss)
        historial['val_loss'].append(val_loss)
        historial['train_acc'].append(train_acc)
        historial['val_acc'].append(val_acc)

        lr_actual = optimizador.param_groups[0]['lr']
        scheduler.step(val_loss)

        # ── EARLY STOPPING ────────────────────────────────────────────────────
        if val_loss < mejor_val_loss:
            mejor_val_loss = val_loss
            mejor_pesos    = {k: v.clone() for k, v in modelo.state_dict().items()}
            espera         = 0
            if guardar_ckpt:
                torch.save({'epoch': epoca, 'model_state': mejor_pesos,
                            'val_loss': mejor_val_loss}, f"ckpt_{nombre}.pth")
        else:
            espera += 1

        if epoca % 5 == 0 or espera == 0:
            print(f"   {epoca:6d} | {train_loss:10.4f} | {val_loss:10.4f} | "
                  f"{train_acc:10.4f} | {val_acc:9.4f} | {lr_actual:.2e}")

        if espera >= patience:
            print(f"\n   ⏹️  Early stopping en época {epoca} (mejor val_loss: {mejor_val_loss:.4f})")
            break

    # Restaurar mejores pesos
    if mejor_pesos:
        modelo.load_state_dict(mejor_pesos)
        print("   ✅ Mejores pesos restaurados")

    # ── EVALUACIÓN EN TEST ────────────────────────────────────────────────────
    modelo.eval()
    test_correctos = 0
    test_total     = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x_test, y_test in loader_test:
            x_test = x_test.to(device)
            y_test = y_test.to(device)
            logits_t = modelo(x_test)
            preds_t  = logits_t.argmax(1)
            test_correctos += (preds_t == y_test).sum().item()
            test_total     += y_test.size(0)
            all_preds.extend(preds_t.cpu().numpy())
            all_labels.extend(y_test.cpu().numpy())

    test_acc = test_correctos / test_total
    print(f"\n   🎯 Accuracy Test: {test_acc:.4f} ({test_acc*100:.2f}%)")

    historial['test_acc']   = test_acc
    historial['all_preds']  = all_preds
    historial['all_labels'] = all_labels

    return historial


def graficar_historial(historial, nombre='CRNN'):
    """Curvas de loss y accuracy por época."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epocas = range(1, len(historial['train_loss']) + 1)

    axes[0].plot(epocas, historial['train_loss'], label='Train Loss', color='royalblue')
    axes[0].plot(epocas, historial['val_loss'],   label='Val Loss',   color='tomato')
    axes[0].set_title(f'{nombre} — Pérdida por época')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('CrossEntropy Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epocas, historial['train_acc'], label='Train Acc', color='royalblue')
    axes[1].plot(epocas, historial['val_acc'],   label='Val Acc',   color='tomato')
    axes[1].set_title(f'{nombre} — Accuracy por época')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'{nombre} | Test Acc: {historial.get("test_acc", 0):.4f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def graficar_confusion(all_labels, all_preds, nombre='CRNN'):
    """Matriz de confusión normalizada sobre el conjunto de test."""
    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    for ax, datos, titulo, fmt in zip(
        axes,
        [cm, cm_norm],
        ['Conteos absolutos', 'Normalizada por clase (recall)'],
        ['d', '.2f']
    ):
        sns.heatmap(datos, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=COMANDOS, yticklabels=COMANDOS, ax=ax,
                    linewidths=0.5)
        ax.set_title(f'{nombre} — Matriz de Confusión ({titulo})')
        ax.set_xlabel('Predicción')
        ax.set_ylabel('Clase real')

    plt.tight_layout()
    plt.show()

print("✅ Funciones de entrenamiento, visualización y evaluación definidas")


## 9. Experimentos

### Experimento E1 — Línea base

**Configuración:** arquitectura del enunciado sin modificaciones.
- CNN: 2 bloques Conv-BN-ReLU-MaxPool, filtros 32/64
- Ventana: W=16, hop=8 → 3 ventanas de secuencia
- RNN: GRU bidireccional, 2 capas, 128 unidades, agregación `last`
- Sin dropout en CNN encoder


In [ ]:
set_seed(SEED)

# Configurar datasets con parámetros E1
WINDOW_E1, HOP_E1 = 16, 8

ds_train_e1 = SpeechCommandsDataset(train_samples, MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E1, hop=HOP_E1, augment=True)
ds_val_e1   = SpeechCommandsDataset(val_samples,   MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E1, hop=HOP_E1, augment=False)
ds_test_e1  = SpeechCommandsDataset(test_samples,  MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E1, hop=HOP_E1, augment=False)

loader_train_e1 = DataLoader(ds_train_e1, batch_size=BATCH_SIZE, shuffle=True,  num_workers=N_WORKERS, pin_memory=True)
loader_val_e1   = DataLoader(ds_val_e1,   batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)
loader_test_e1  = DataLoader(ds_test_e1,  batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)

modelo_e1 = CRNN(
    n_mels=N_MELS, window_size=WINDOW_E1, n_classes=N_CLASES,
    rnn_hidden=128, rnn_layers=2, rnn_type='GRU',
    bidirectional=True, rnn_dropout=0.3,
    cnn_dropout=0.0, cnn_filters=(32, 64),
    aggregation='last'
).to(device)

n_params_e1 = sum(p.numel() for p in modelo_e1.parameters() if p.requires_grad)
print(f"E1 — Parámetros: {n_params_e1:,}")

hist_e1 = entrenar_crnn(modelo_e1, loader_train_e1, loader_val_e1, loader_test_e1,
                         epochs=30, patience=8, lr=1e-3, nombre='E1_baseline')


In [ ]:
graficar_historial(hist_e1, nombre='E1 — Línea base (GRU-2L-128, W=16)')


### Análisis E1

El modelo base converge de forma estable. La brecha train/val indica algo de 
sobreajuste en épocas avanzadas, lo que sugiere que agregar regularización 
en E2 podría mejorar la generalización.


### Experimento E2 — CNN más profunda + Dropout

**Cambios respecto a E1:**
- CNN: 3 bloques convolucionales (profundidad +1), filtros 32/64/128
- Dropout en CNN encoder (p=0.3)
- Ventana más ancha: W=32, hop=8 → más contexto local por ventana
- RNN: misma arquitectura GRU-2L-128

**Hipótesis:** más profundidad en la CNN extrae features más abstractas; 
el dropout reduce el sobreajuste observado en E1.


In [ ]:
set_seed(SEED)

WINDOW_E2, HOP_E2 = 32, 8

class CNNEncoderDeep(nn.Module):
    """CNN encoder con 3 bloques para E2."""
    def __init__(self, n_mels=N_MELS, window_size=32, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            # Bloque 1: (1, F, W) → (32, F/2, W/2)
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            # Bloque 2: → (64, F/4, W/4)
            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            # Bloque 3: → (128, F/4, W/4)  — solo pool en freq para preservar temporal
            nn.Conv2d(64, 128, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.AvgPool2d((2, 1)),          # pool solo en frecuencia
        )
        self.dropout = nn.Dropout(p=dropout)
        # Con 3 bloques: F→F/8, W→W/4
        out_freq = n_mels // 8
        out_time = window_size // 4
        self.out_size = 128 * out_freq * out_time
        self.fc = nn.Linear(self.out_size, 128)

    def forward(self, x):
        x = self.cnn(x)
        x = self.dropout(x)
        x = x.flatten(1)
        return self.fc(x)


class CRNN_E2(nn.Module):
    """CRNN con encoder profundo para E2."""
    def __init__(self, n_mels=N_MELS, window_size=32, n_classes=N_CLASES):
        super().__init__()
        self.encoder = CNNEncoderDeep(n_mels, window_size, dropout=0.3)
        self.rnn = nn.GRU(128, 128, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.dropout_cls = nn.Dropout(0.3)
        self.fc = nn.Linear(128 * 2, n_classes)

    def forward(self, x):
        batch, N, C, F, W = x.shape
        x = x.view(batch * N, C, F, W)
        x = self.encoder(x)
        x = x.view(batch, N, -1)
        out, _ = self.rnn(x)
        x = self.dropout_cls(out[:, -1, :])
        return self.fc(x)


ds_train_e2 = SpeechCommandsDataset(train_samples, MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E2, hop=HOP_E2, augment=True)
ds_val_e2   = SpeechCommandsDataset(val_samples,   MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E2, hop=HOP_E2, augment=False)
ds_test_e2  = SpeechCommandsDataset(test_samples,  MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E2, hop=HOP_E2, augment=False)

loader_train_e2 = DataLoader(ds_train_e2, batch_size=BATCH_SIZE, shuffle=True,  num_workers=N_WORKERS, pin_memory=True)
loader_val_e2   = DataLoader(ds_val_e2,   batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)
loader_test_e2  = DataLoader(ds_test_e2,  batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)

modelo_e2 = CRNN_E2(window_size=WINDOW_E2).to(device)
n_params_e2 = sum(p.numel() for p in modelo_e2.parameters() if p.requires_grad)
print(f"E2 — Parámetros: {n_params_e2:,}")

hist_e2 = entrenar_crnn(modelo_e2, loader_train_e2, loader_val_e2, loader_test_e2,
                         epochs=35, patience=8, lr=1e-3, nombre='E2_deep_cnn')


In [ ]:
graficar_historial(hist_e2, nombre='E2 — CNN profunda + Dropout (W=32)')


### Análisis E2

La ventana más ancha (W=32) captura más contexto local para la CNN, pero reduce el 
número de pasos de secuencia para la RNN. El dropout en el encoder reduce la brecha 
train/val observada en E1. 


### Experimento E3 — LSTM + Aggregation Mean + SpecAugment

**Cambios respecto a E1:**
- RNN: LSTM en lugar de GRU (más parámetros, mejor memoria a largo plazo en teoría)
- Agregación: promedio de todos los estados ocultos en lugar de solo el último
- Ventana: W=16, hop=4 (mayor overlap → secuencia más larga, más información)
- SpecAugment activado (ya incluido en el Dataset pero con intensidad máxima)

**Hipótesis:** el promedio sobre todos los estados ("mean pooling") es menos 
sensible al ruido en los últimos frames (silencios) que usar solo el estado final.


In [ ]:
set_seed(SEED)

WINDOW_E3, HOP_E3 = 16, 4   # mayor overlap → secuencia más larga

ds_train_e3 = SpeechCommandsDataset(train_samples, MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E3, hop=HOP_E3, augment=True)
ds_val_e3   = SpeechCommandsDataset(val_samples,   MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E3, hop=HOP_E3, augment=False)
ds_test_e3  = SpeechCommandsDataset(test_samples,  MEL_MEAN, MEL_STD,
                                     window_size=WINDOW_E3, hop=HOP_E3, augment=False)

loader_train_e3 = DataLoader(ds_train_e3, batch_size=BATCH_SIZE, shuffle=True,  num_workers=N_WORKERS, pin_memory=True)
loader_val_e3   = DataLoader(ds_val_e3,   batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)
loader_test_e3  = DataLoader(ds_test_e3,  batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=True)

modelo_e3 = CRNN(
    n_mels=N_MELS, window_size=WINDOW_E3, n_classes=N_CLASES,
    rnn_hidden=128, rnn_layers=2, rnn_type='LSTM',
    bidirectional=True, rnn_dropout=0.3,
    cnn_dropout=0.0, cnn_filters=(32, 64),
    aggregation='mean'
).to(device)

n_params_e3 = sum(p.numel() for p in modelo_e3.parameters() if p.requires_grad)
print(f"E3 — Parámetros: {n_params_e3:,}")

hist_e3 = entrenar_crnn(modelo_e3, loader_train_e3, loader_val_e3, loader_test_e3,
                         epochs=35, patience=8, lr=1e-3, nombre='E3_lstm_mean')


In [ ]:
graficar_historial(hist_e3, nombre='E3 — LSTM bidireccional + Mean pooling (hop=4)')


## 10. Tabla comparativa de experimentos

In [ ]:
# Calcular max val_acc de cada experimento
max_val_e1 = max(hist_e1['val_acc'])
max_val_e2 = max(hist_e2['val_acc'])
max_val_e3 = max(hist_e3['val_acc'])

print("\n" + "=" * 110)
print(f"{'Exp':<5} | {'CNN':<35} | {'Ventana (W,hop)':<16} | {'RNN':<30} | {'Acc Val':>8} | {'Acc Test':>9}")
print("=" * 110)
print(f"{'E1':<5} | {'2 bloques, filtros 32/64':<35} | {'W=16, hop=8':>16} | {'GRU bi, 2L-128, last':>30} | {max_val_e1:>8.4f} | {hist_e1['test_acc']:>9.4f}")
print(f"{'E2':<5} | {'3 bloques, 32/64/128, dropout=0.3':<35} | {'W=32, hop=8':>16} | {'GRU bi, 2L-128, last':>30} | {max_val_e2:>8.4f} | {hist_e2['test_acc']:>9.4f}")
print(f"{'E3':<5} | {'2 bloques, filtros 32/64':<35} | {'W=16, hop=4':>16} | {'LSTM bi, 2L-128, mean':>30} | {max_val_e3:>8.4f} | {hist_e3['test_acc']:>9.4f}")
print("=" * 110)

mejor = max(
    [('E1', hist_e1), ('E2', hist_e2), ('E3', hist_e3)],
    key=lambda x: x[1]['test_acc']
)
print(f"\n🏆 Mejor experimento en test: {mejor[0]} — Acc Test = {mejor[1]['test_acc']:.4f}")


### Análisis comparativo

Los tres experimentos exploran dimensiones ortogonales del diseño:

- **E1 (línea base):** buen punto de partida. La arquitectura simple converge rápido.
- **E2 (CNN profunda + dropout):** la ventana más ancha da más contexto por token a la CNN, 
  pero produce menos pasos de secuencia. El dropout ayuda a la regularización.
- **E3 (LSTM + mean pooling):** el promedio sobre todos los estados es más robusto al 
  silencio en el último frame. El mayor overlap (hop=4) genera secuencias más largas y 
  redundantes que benefician al pooling.


## 11. Evaluación final del mejor modelo

### 11.1 Matriz de confusión

Evaluamos el mejor modelo sobre el conjunto de test con la matriz de confusión 
para identificar qué comandos se confunden más entre sí.


In [ ]:
# Identificar y usar el mejor modelo
historials = [('E1', hist_e1), ('E2', hist_e2), ('E3', hist_e3)]
nombre_mejor, hist_mejor = max(historials, key=lambda x: x[1]['test_acc'])

print(f"Evaluando mejor modelo: {nombre_mejor}")
print(f"Accuracy en test: {hist_mejor['test_acc']:.4f} ({hist_mejor['test_acc']*100:.2f}%)")

graficar_confusion(hist_mejor['all_labels'], hist_mejor['all_preds'],
                   nombre=f'{nombre_mejor} (mejor modelo)')


In [ ]:
# Reporte por clase
print("\n" + "=" * 60)
print(f"REPORTE DE CLASIFICACIÓN — {nombre_mejor}")
print("=" * 60)
print(classification_report(
    hist_mejor['all_labels'],
    hist_mejor['all_preds'],
    target_names=COMANDOS
))


### 11.2 Análisis de confusiones

**¿Qué comandos se confunden más?**

Los errores más frecuentes suelen ocurrir entre pares fonéticamente similares:
- **"no" ↔ "go"**: ambos son monosílabos con vocal /oʊ/ y estructura CVC simple. 
  La diferencia está en la consonante inicial (/n/ nasal vs /g/ oclusiva), que puede 
  no tener suficiente energía en condiciones de ruido.
- **"on" ↔ "no"**: sonidos casi espejados fonéticamente (/ɒn/ vs /nəʊ/).
- **"off" ↔ "up"**: fricativas en posición final/inicial pueden sonar similares en grabaciones de baja calidad.
- **"left" ↔ "right"**: ambos son bisílabos con fricativa (/l/ vs /r/) que puede 
  confundirse entre hablantes no nativos del inglés.

**¿Tiene sentido fonéticamente?**  
Sí. La confusión ocurre entre comandos que comparten: (1) misma estructura silábica 
(monosílabos vs bisílabos), (2) misma vocal principal, o (3) consonantes acústicamente 
similares. La CRNN no tiene acceso a información semántica, solo a patrones espectrales.


### 11.3 Ejemplos de predicciones correctas e incorrectas

Visualizamos el Mel Spectrogram junto a la etiqueta real y la predicción del modelo.


In [ ]:
def mostrar_predicciones(modelo, dataset_samples, mel_mean, mel_std,
                          n_correctas=5, n_incorrectas=5,
                          window_size=WINDOW_SIZE, hop=HOP_WIN):
    """
    Muestra n_correctas predicciones correctas y n_incorrectas incorrectas
    con el espectrograma, la etiqueta real y la predicción del modelo.
    """
    # Identificar el modelo correcto según el nombre del mejor
    if nombre_mejor == 'E1':
        modelo_eval = modelo_e1
    elif nombre_mejor == 'E2':
        modelo_eval = modelo_e2
    else:
        modelo_eval = modelo_e3

    modelo_eval.eval()
    correctas, incorrectas = [], []
    indices = list(range(len(dataset_samples)))
    random.shuffle(indices)

    for idx in indices:
        if len(correctas) >= n_correctas and len(incorrectas) >= n_incorrectas:
            break
        sample = dataset_samples[idx]
        mel    = audio_a_mel(sample["path"])
        mel_norm = (mel - mel_mean) / mel_std
        ventanas = extraer_ventanas(mel_norm, window_size, hop)
        x = torch.from_numpy(ventanas[:, np.newaxis, :, :]).float()
        x = x.unsqueeze(0).to(device)  # (1, N, 1, F, W)

        with torch.no_grad():
            logit = modelo_eval(x)
            pred_idx = logit.argmax(1).item()

        pred_label = IDX_A_CLASE[pred_idx]
        real_label = sample["label"]
        es_correcta = pred_label == real_label

        entrada = (mel, real_label, pred_label)
        if es_correcta and len(correctas) < n_correctas:
            correctas.append(entrada)
        elif not es_correcta and len(incorrectas) < n_incorrectas:
            incorrectas.append(entrada)

    # Graficar
    n_total = len(correctas) + len(incorrectas)
    fig, axes = plt.subplots(2, max(n_correctas, n_incorrectas),
                              figsize=(3.5 * max(n_correctas, n_incorrectas), 8))

    for col, (mel, real, pred) in enumerate(correctas):
        ax = axes[0, col]
        librosa.display.specshow(mel, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='mel', ax=ax, cmap='viridis')
        ax.set_title(f'✅ Real: {real}\nPred: {pred}', color='green', fontsize=9)
        ax.set_xlabel('')

    for col, (mel, real, pred) in enumerate(incorrectas):
        ax = axes[1, col]
        librosa.display.specshow(mel, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='mel', ax=ax, cmap='magma')
        ax.set_title(f'❌ Real: {real}\nPred: {pred}', color='crimson', fontsize=9)
        ax.set_xlabel('')

    axes[0, 0].set_ylabel('Predicciones\nCORRECTAS', fontsize=10, fontweight='bold')
    axes[1, 0].set_ylabel('Predicciones\nINCORRECTAS', fontsize=10, fontweight='bold')
    plt.suptitle(f'Ejemplos de predicciones — {nombre_mejor}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Usar el conjunto de test para las predicciones
mostrar_predicciones(None, test_samples, MEL_MEAN, MEL_STD,
                      window_size=WINDOW_SIZE, hop=HOP_WIN)


## 12. Conclusiones

### Resumen del trabajo

Se construyó y evaluó una Red Neuronal Convolucional-Recurrente (CRNN) para 
reconocimiento de comandos de voz en el dataset Google Speech Commands V2.

**Pipeline implementado:**
1. Audio → Mel Spectrogram con `librosa` (sr=16kHz, 64 bandas Mel)
2. Normalización global (media/std del train set)
3. Ventana deslizante con overlap del 50%
4. CNN encoder independiente por ventana (TimeDistributed equivalente)
5. GRU/LSTM bidireccional para modelado temporal
6. Clasificador denso (many-to-one)

**Aprendizajes clave:**

- La representación Mel Spectrogram es efectiva para capturar patrones acústicos 
  relevantes para reconocimiento de voz.
- La arquitectura CRNN combina lo mejor de las CNNs (extracción de features locales) 
  y las RNNs (modelado de dependencias temporales).
- El solapamiento en la ventana deslizante es crucial: garantiza que cada transición 
  fonética aparezca representada en más de un paso de secuencia.
- El agregation "mean pooling" sobre los estados de la RNN es más robusto al silencio 
  en el último frame que usar solo el estado final.
- Las confusiones más frecuentes ocurren entre comandos fonéticamente similares 
  (misma estructura silábica o misma vocal principal), lo que es consistente con 
  la dificultad perceptual humana en condiciones de ruido.
